# 04 · Agentic RAG with LangGraph
The full agent: **route → rewrite → retrieve → grade → (refine loop) → generate → verify → finalize**, with bounded retries and a fail-closed `no_answer` path.

In [ ]:
# ---- Shared setup (identical in every notebook) -------------------------------
import os, sys, json, pathlib, time
from dotenv import load_dotenv
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "sample_data"))

# Automatically load environment variables from rag_service/.env and repo-root .env
load_dotenv(ROOT / ".env", override=False)
load_dotenv(ROOT.parent / ".env", override=False)

# OFFLINE=True  -> deterministic hash embeddings + mock LLM + local Chroma. No keys, no network.
# OFFLINE=False -> REAL Gemini/Groq/OpenAI + Chroma Cloud from your .env (default when keys are present)
has_keys = bool(os.getenv("GEMINI_API_KEY") or os.getenv("GROQ_API_KEY") or os.getenv("OPENAI_API_KEY"))
OFFLINE = (os.getenv("NB_OFFLINE", "0" if has_keys else "1") == "1")
if OFFLINE:
    os.environ["RAG_OFFLINE"] = "1"
    os.environ["CHROMA_LOCAL_DIR"] = str(ROOT / "chroma_notebook")   # shared by all notebooks
else:
    os.environ["RAG_OFFLINE"] = "0"

from app.config import get_settings, reset_settings_cache
reset_settings_cache()
S = get_settings()
print("mode            :", "OFFLINE (mock)" if S.offline else "LIVE providers")
print("vector store    :", "Chroma Cloud" if S.use_chroma_cloud else f"local ({S.chroma_local_dir})")
print("embedding       :", "hash512" if S.offline else S.embedding_provider)
print("LLM providers   :", [p for p, k in (("gemini", S.gemini_key), ("groq", S.groq_key), ("openai", S.openai_key)) if k] or "none (offline mock)")

## Graph structure

In [ ]:
from app.agent import get_graph
print(get_graph().get_graph().draw_mermaid())

Paste the Mermaid text into https://mermaid.live to see the diagram.

In [ ]:
from app.agent import ask
from app.store import get_store
assert get_store().count() > 0, "Run notebook 01 first"
def run(q, subj="BIO101", history=None):
    out = ask(q, subject_codes=[subj], history=history or [])
    print("Q:", q); print("A:", out["answer"])
    print(f"   confidence={out['confidence']} grounded={out['grounded']} intent={out['intent']} latency={out['latency_ms']}ms")
    print("   sources:", [(s['file_name'], 'p.%s' % s['page']) for s in out["sources"]])
    print("   path   :", " → ".join(t["node"] for t in out["agent_trace"]), "\n")
    return out
run("hello")
run("Which organelle produces ATP?")
run("Where does photosynthesis take place in a plant cell?")
run("Who won the football world cup in 1998?")
run("what is the weather today")

## Multi-turn: follow-up question resolved with chat history

In [ ]:
h = [{"role":"user","content":"What is DNA replication?"},{"role":"assistant","content":"It copies DNA before cell division."}]
o = run("Which enzyme does it use?", history=h)
print("rewritten:", o["standalone_question"])

## Scope enforcement: same question, other subject → refusal

In [ ]:
run("Which organelle produces ATP?", subj="PHY101")

## Inspect the full trace of one run

In [ ]:
import pandas as pd
pd.DataFrame(ask("Which enzyme adds nucleotides during DNA replication?", subject_codes=["BIO101"])["agent_trace"]).drop(columns="t")